# Meal Planning Agent — Prompt Experiments

Scratchpad for iterating on prompts with the OpenAI Agents SDK before building the app.

**Setup:** copy `.env.example` to `.env` in the project root and add your `OPENAI_API_KEY`, then launch this notebook with:

```bash
uv run jupyter lab
```

In [ ]:
import os

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())  # searches parent dirs, so it finds the project-root .env

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not set — create a .env file in the project root"
print("API key loaded ✔")

## Basic agent

A minimal agent to sanity-check the setup. Jupyter supports top-level `await`, so `Runner.run` can be awaited directly.

In [ ]:
from agents import Agent, Runner

agent = Agent(
    name="Meal Planner",
    instructions=(
        "You are a meal planning assistant. Suggest simple, practical meals "
        "based on the user's preferences and constraints."
    ),
    model="gpt-4.1-mini",
)

result = await Runner.run(agent, "Suggest three vegetarian dinners for this week.")
print(result.final_output)

## Prompt iteration helper

`try_prompt` builds a throwaway agent so different instruction variants can be compared side by side.

In [ ]:
async def try_prompt(instructions: str, user_input: str, model: str = "gpt-4.1-mini") -> str:
    agent = Agent(name="Experiment", instructions=instructions, model=model)
    result = await Runner.run(agent, user_input)
    return result.final_output

In [ ]:
USER_INPUT = "Plan my dinners for Monday to Friday. I'm cooking for two and want to keep the shopping list short."

variant_a = "You are a meal planning assistant. Be concise."

variant_b = (
    "You are a meal planning assistant.\n"
    "- Plan meals that share ingredients to minimise the shopping list.\n"
    "- For each meal give a name, a one-line description, and prep time.\n"
    "- Finish with a consolidated shopping list grouped by supermarket aisle."
)

print("=== Variant A ===")
print(await try_prompt(variant_a, USER_INPUT))
print()
print("=== Variant B ===")
print(await try_prompt(variant_b, USER_INPUT))

## Structured output

Setting `output_type` to a Pydantic model makes the agent return typed data instead of free text — useful once the app needs to consume the plan programmatically.

In [ ]:
from pydantic import BaseModel


class Meal(BaseModel):
    day: str
    name: str
    description: str
    prep_time_minutes: int


class MealPlan(BaseModel):
    meals: list[Meal]
    shopping_list: list[str]


structured_agent = Agent(
    name="Structured Meal Planner",
    instructions="Plan meals that share ingredients to keep the shopping list short.",
    model="gpt-4.1-mini",
    output_type=MealPlan,
)

result = await Runner.run(structured_agent, "Plan vegetarian dinners for Monday to Wednesday, for two people.")
plan = result.final_output

for meal in plan.meals:
    print(f"{meal.day}: {meal.name} ({meal.prep_time_minutes} min) — {meal.description}")
print("\nShopping list:", ", ".join(plan.shopping_list))